# Módulo 2: Curvas Elípticas

## 2.1 La Ecuación de la Curva

Una curva elíptica sobre un cuerpo $\mathbb{F}$ se define por la **ecuación de Weierstrass**:

$$y^2 = x^3 + ax + b$$

con la restricción de que $4a^3 + 27b^2 \neq 0$ (no singular — sin cúspides ni autointersecciones).

### La curva de Bitcoin: secp256k1

$$y^2 = x^3 + 7 \pmod{P}$$

donde $a = 0$, $b = 7$ (una **curva de Koblitz** — el cero en $a$ permite optimizaciones).

### El punto generador $G$

La curva tiene $\approx 2^{256}$ puntos. Para usar criptografía, todos necesitan
acordar un único **punto de partida** — un punto específico $G$ en la curva
que es públicamente conocido y está codificado en cada implementación de Bitcoin.

¿Por qué? Porque la función unidireccional es $P = d \times G$ — eliges un número
secreto $d$ (tu clave privada), multiplicas $G$ por $d$ usando suma de puntos, y
obtienes tu clave pública $P$. Todos usan el **mismo** $G$, así cualquiera puede
verificar tu clave pública y firmas.

$G$ es simplemente un punto $(x, y)$ en la curva. Fue elegido como parte del
estándar secp256k1 — no hay nada matemáticamente especial en comparación con otros
puntos, pero todos deben usar el mismo. Genera el grupo entero: $G, 2G, 3G,
\ldots, NG = \mathcal{O}$ recorre los $N$ puntos y vuelve al infinito.
Por eso se llama **generador** — cada punto en la curva es algún múltiplo de $G$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import hashlib
import secrets
import numpy as np
import matplotlib.pyplot as plt

from ecc import *

## 2.2 Curvas Elípticas Sobre Cuerpos Finitos

Sobre los números reales, una curva elíptica es una curva suave. Sobre un **cuerpo
finito** $\mathbb{F}_p$, se convierte en un conjunto discreto de puntos — hay
exactamente $N$ de ellos (más el punto en el infinito).

Visualicemos una curva elíptica pequeña para construir intuición antes de trabajar
con los números enormes de secp256k1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

a_small, b_small = 1, 1
p_small = 23

def is_quadratic_residue(n, p):
    if n % p == 0:
        return True
    return pow(n, (p - 1) // 2, p) == 1

def mod_sqrt_small(a, p):
    return pow(a, (p + 1) // 4, p)

points = []
for x in range(p_small):
    rhs = (x**3 + a_small * x + b_small) % p_small
    if is_quadratic_residue(rhs, p_small):
        y = mod_sqrt_small(rhs, p_small)
        points.append((x, y))
        if y != 0 and y != p_small - y:
            points.append((x, p_small - y))

points.sort()
xs_fp, ys_fp = zip(*points)

# --- Comparación: curva real vs cuerpo finito ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Izquierda: curva suave sobre los reales
x_real = np.linspace(-1.5, 4, 1000)
rhs_real = x_real**3 + a_small * x_real + b_small
mask = rhs_real >= 0
y_pos = np.sqrt(rhs_real[mask])
ax1.plot(x_real[mask], y_pos, 'b-', linewidth=2)
ax1.plot(x_real[mask], -y_pos, 'b-', linewidth=2)
ax1.axhline(0, color='gray', linewidth=0.5)
ax1.axvline(0, color='gray', linewidth=0.5)
ax1.set_title(f'$y^2 = x^3 + {a_small}x + {b_small}$ sobre $\\mathbb{{R}}$', fontsize=14)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

# Derecha: diagrama de dispersión sobre F_p
ax2.scatter(xs_fp, ys_fp, c='blue', s=40, zorder=5)
ax2.set_title(f'$y^2 = x^3 + {a_small}x + {b_small}$ sobre $\\mathbb{{F}}_{{{p_small}}}$\n'
              f'{len(points)} puntos + punto en el infinito', fontsize=14)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_xlim(-1, p_small)
ax2.set_ylim(-1, p_small)
ax2.axhline(p_small / 2, color='red', linewidth=0.8, linestyle='--', alpha=0.5, label=f'y = {p_small}/ 2 (simetría)')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n{len(points)} puntos en la curva (+ punto en el infinito = {len(points) + 1})")

### Observación

Nota la **simetría vertical** — para cada punto $(x, y)$ hay un punto $(x, p-y)$.
Esto es porque si $y^2 \equiv c \pmod{p}$, entonces $(-y)^2 \equiv c \pmod{p}$
también. La negación de un punto es su reflejo vertical: $-P = (x, -y)$.

---